# 3 National Flood Simulation Scenario

This analysis notebook loads a baseline basin risk table and a scenario basin
risk table, builds paired curve dictionaries, and runs the flood simulation.


In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd

from sovereign.flood import (
    BasinLossCurve,
    build_basin_curves,
    extract_sectoral_losses,
    run_simulation,
)


In [7]:
# USER CONFIG
model = "wri"
n_years = 10000
scenario_mode = "combined"
scenario_name = "pubinf_urban_mask_50pct_nbs_shift_150pct"

root = Path.cwd().parent
baseline_basin_path = root / "outputs" / "flood" / "risk" / "basins" / f"risk_basins_m-{model}.csv"
scenario_basin_path = root / "outputs" / "flood" / "adaptation" / scenario_mode / scenario_name / "basins" / f"risk_basins_m-{model}.csv"
copula_path = root / "outputs" / "flood" / "dependence" / "copulas" / "copula_random_numbers.gzip"


In [8]:
# Load basin risk tables
baseline_risk_data = pd.read_csv(baseline_basin_path)
baseline_risk_data = baseline_risk_data.iloc[:, 1:] if "Unnamed: 0" in baseline_risk_data.columns[0] else baseline_risk_data
baseline_risk_data["AEP"] = 1 / baseline_risk_data["RP"]
baseline_risk_data["Pr_L_AEP"] = np.where(baseline_risk_data["Pr_L"] == 0, 0, 1 / baseline_risk_data["Pr_L"])
baseline_risk_data.reset_index(drop=True, inplace=True)

scenario_risk_data = pd.read_csv(scenario_basin_path)
scenario_risk_data = scenario_risk_data.iloc[:, 1:] if "Unnamed: 0" in scenario_risk_data.columns[0] else scenario_risk_data
if "AEP" not in scenario_risk_data.columns:
    scenario_risk_data["AEP"] = 1 / scenario_risk_data["RP"]
if "Pr_L_AEP" not in scenario_risk_data.columns:
    scenario_risk_data["Pr_L_AEP"] = np.where(scenario_risk_data["Pr_L"] == 0, 0, 1 / scenario_risk_data["Pr_L"])
scenario_risk_data.reset_index(drop=True, inplace=True)

copula_random_numbers = pd.read_parquet(copula_path).iloc[:n_years].copy()

baseline_risk_data.head()


,FID,GID_1,NAME,HB_L6,Pr_L,damages,adapted_damages,RP,Sector,AEP,Pr_L_AEP
0,0,UGA.3_1,Arua,1.061054e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5
1,1,UGA.47_1,Nebbi,1.061054e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5
2,2,UGA.27_1,Kitgum,1.060999e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5
3,3,UGA.41_1,Moyo,1.061033e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5
4,4,UGA.3_1,Arua,1.061033e+09,2.0,6160.324707,6160.324707,5,Public,0.2,0.5


In [9]:
# Build baseline and scenario curves
baseline_curves: dict[int, BasinLossCurve] = build_basin_curves(baseline_risk_data)
scenario_curves: dict[int, BasinLossCurve] = build_basin_curves(scenario_risk_data)


In [10]:
# Run paired simulation
baseline_losses, scenario_losses = run_simulation(
    baseline_curves,
    scenario_curves,
    n_years,
    copula_random_numbers,
)


100%|███████████████████████████████████████████████████████████████████████████| 10000/10000 [01:28<00:00, 112.89it/s]


In [11]:
# Compare sectoral losses
baseline_sectoral_loss = extract_sectoral_losses(baseline_losses, n_years)
scenario_sectoral_loss = extract_sectoral_losses(scenario_losses, n_years)

comparison_df = pd.DataFrame({
    "metric": ["GVA_loss", "CAP_dam", "AGR_loss", "MAN_loss", "SER_loss", "PUB_dam", "PRI_dam"],
    "baseline_aal": [baseline_sectoral_loss[c].mean() for c in ["GVA_loss", "CAP_dam", "AGR_loss", "MAN_loss", "SER_loss", "PUB_dam", "PRI_dam"]],
    "scenario_aal": [scenario_sectoral_loss[c].mean() for c in ["GVA_loss", "CAP_dam", "AGR_loss", "MAN_loss", "SER_loss", "PUB_dam", "PRI_dam"]],
})
comparison_df["aal_change"] = comparison_df["scenario_aal"] - comparison_df["baseline_aal"]
comparison_df["pct_change"] = np.where(
    comparison_df["baseline_aal"] != 0,
    100 * comparison_df["aal_change"] / comparison_df["baseline_aal"],
    np.nan,
)
comparison_df


,metric,baseline_aal,scenario_aal,aal_change,pct_change
0,GVA_loss,4.615911e+07,4.603566e+07,-1.234467e+05,-0.267437
1,CAP_dam,5.597622e+07,5.319610e+07,-2.780119e+06,-4.966608
2,AGR_loss,1.498612e+07,1.486589e+07,-1.202280e+05,-0.802263
3,MAN_loss,1.024031e+07,1.023930e+07,-1.006460e+03,-0.009828
4,SER_loss,2.093268e+07,2.093047e+07,-2.212171e+03,-0.010568
5,PUB_dam,2.079314e+07,1.817793e+07,-2.615203e+06,-12.577240
6,PRI_dam,3.518308e+07,3.501817e+07,-1.649167e+05,-0.468739
